# Modelo preditivo de análise de risco de defasagem

## Objetivo
Este notebook tem como objetivo treinar e avaliar um modelo preditivo para classificação do risco de defasagem dos alunos, utilizando a base processada reduzida para Machine Learning.

As etapas contempladas são:
- feature engineering;
- separação dos dados em treino e teste;
- modelagem preditiva;
- avaliação dos resultados.

### 2) Importações

In [20]:
from pathlib import Path
import sys

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
)

ROOT_DIR = Path.cwd().resolve().parents[0] if "notebooks" in str(Path.cwd()) else Path.cwd().resolve()
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

from src.features import (
    FEATURE_COLUMNS,
    TARGET_COLUMN,
    prepare_training_features,
)

from src.model import (
    load_training_data,
    build_pipeline,
    train_model,
)

### 3) Carregar a base de treino

In [21]:
training_path = Path("../data/processed/base_processada_reduzida_ML.parquet")
df = load_training_data(training_path)

print(df.shape)
display(df.head())
print(df.dtypes)

(3030, 18)


,ra,ano_pede,inde,n_av,iaa,ieg,ips,ipp,ida,mat,por,ing,ipv,ian,fase_ideal,defasagem,defasagem_ano_seguinte,risco_defasagem_futuro
0,RA-1,2022,5.783000,4.0,8.3,4.1,5.6,NaN,4.0,2.7,3.5,6.0,7.278,5.0,Fase 8 (Universitários),-1,0.0,1
1,RA-1,2023,5.783000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,10.0,None,0,0.0,0
2,RA-1,2024,5.783133,0.0,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,10.0,None,0,NaN,<NA>
3,RA-10,2022,5.784000,4.0,8.3,5.2,5.0,NaN,4.1,3.3,2.6,6.4,7.056,5.0,Fase 8 (Universitários),-1,NaN,<NA>
4,RA-100,2022,7.618000,4.0,8.8,7.8,5.0,NaN,7.6,7.0,7.8,8.1,7.250,10.0,Fase 3 (7º e 8º ano),1,NaN,<NA>


ra                        string[python]
ano_pede                           int64
inde                             float64
n_av                             float64
iaa                              float64
ieg                              float64
ips                              float64
ipp                              float64
ida                              float64
mat                              float64
por                              float64
ing                              float64
ipv                              float64
ian                              float64
fase_ideal                        object
defasagem                          int64
defasagem_ano_seguinte           float64
risco_defasagem_futuro             Int64
dtype: object


### 4) Entender a base

In [22]:
print("Colunas da base:")
print(df.columns.tolist())

print("Distribuição da variável alvo:")
print(df[TARGET_COLUMN].value_counts(dropna=False))
print(df[TARGET_COLUMN].value_counts(normalize=True, dropna=False))

print("Valores ausentes por coluna:")
display(df.isna().sum().sort_values(ascending=False))

Colunas da base:
['ra', 'ano_pede', 'inde', 'n_av', 'iaa', 'ieg', 'ips', 'ipp', 'ida', 'mat', 'por', 'ing', 'ipv', 'ian', 'fase_ideal', 'defasagem', 'defasagem_ano_seguinte', 'risco_defasagem_futuro']
Distribuição da variável alvo:


KeyError: 'risco_defasagem_atual'

### 5) Feature engineering
Mostrar claramente que está preparando as variáveis para treino.

## Feature engineering

Nesta etapa, as variáveis da base de modelagem foram organizadas conforme o conjunto de atributos definido para o modelo.

As principais ações realizadas foram:
- seleção das colunas preditoras relevantes;
- coerção de tipos numéricos;
- padronização das variáveis categóricas;
- remoção de registros sem variável alvo válida.

O pré-processamento completo utilizado pelo modelo é executado por meio de pipeline, incluindo:
- imputação de valores ausentes;
- padronização de variáveis numéricas;
- codificação one-hot para variáveis categóricas.

In [ ]:
X, y = prepare_training_features(df, target_column=TARGET_COLUMN)

print("Shape de X:", X.shape)
print("Shape de y:", y.shape)

display(X.head())
display(y.head())

## 6) Separação treino e teste

### Separação dos dados em treino e teste

A base foi separada em:
- 80% para treino
- 20% para teste

Foi utilizada estratificação pela variável alvo, quando aplicável, para preservar a distribuição das classes entre os conjuntos.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y if y.nunique() > 1 else None,
)

print("Treino:", X_train.shape, y_train.shape)
print("Teste:", X_test.shape, y_test.shape)

## 7) Modelagem preditiva

### Modelagem preditiva

Foi utilizado um pipeline composto por:
- imputação de valores ausentes;
- padronização de atributos numéricos;
- codificação de atributos categóricos;
- algoritmo de regressão logística para classificação do risco de defasagem.

## Por que escolhemos Regressão Logística

A regressão logística foi utilizada como primeiro modelo por vários motivos importantes:

- **Interpretabilidade**: permite entender claramente como cada variável influencia o risco de defasagem.
- **Baseline robusto**: é um modelo clássico para classificação binária e serve como referência para modelos mais complexos.
- **Boa adequação para classificação binária**: o problema é prever risco de defasagem (sim/não), que é naturalmente binário.
- **Facilidade de deploy no Streamlit**: modelo leve, rápido e fácil de serializar com pickle/joblib, ideal para aplicações interativas.

In [ ]:
pipeline = build_pipeline()
pipeline.fit(X_train, y_train)

print("Modelo treinado com sucesso.")

### 8) Predições

In [ ]:
y_pred = pipeline.predict(X_test)
print("Primeiras predições:")
print(y_pred[:10])

In [ ]:
# Se houver predict_proba
if hasattr(pipeline, "predict_proba"):
    y_proba = pipeline.predict_proba(X_test)
    display(pd.DataFrame(y_proba).head())

### 9) Avaliação do modelo

In [ ]:
acc = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred, zero_division=0)
rec = recall_score(y_test, y_pred, zero_division=0)
f1 = f1_score(y_test, y_pred, zero_division=0)

print(f"Acurácia:  {acc:.4f}")
print(f"Precisão:  {prec:.4f}")
print(f"Recall:    {rec:.4f}")
print(f"F1-score:  {f1:.4f}")

In [ ]:
# Se binário:
if hasattr(pipeline, "predict_proba") and y.nunique() == 2:
    y_score = pipeline.predict_proba(X_test)[:, 1]
    auc = roc_auc_score(y_test, y_score)
    print(f"ROC AUC:   {auc:.4f}")

In [ ]:
# Matriz de confusão
cm = confusion_matrix(y_test, y_pred)
cm_df = pd.DataFrame(cm)
display(cm_df)

In [ ]:
# Relatório de classificação
report = classification_report(y_test, y_pred, zero_division=0)
print(report)

### 10) Avaliação mais organizada em dataframe

In [ ]:
metricas = {
    "metrica": ["accuracy", "precision", "recall", "f1"],
    "valor": [acc, prec, rec, f1],
}

if hasattr(pipeline, "predict_proba") and y.nunique() == 2:
    metricas["metrica"].append("roc_auc")
    metricas["valor"].append(auc)

metricas_df = pd.DataFrame(metricas)
display(metricas_df)

### 11) Treinar usando sua função pronta e salvar artefatos
Depois de mostrar o processo manual, construo o pipeline oficial do projeto.
Isso vai gerar:

data/artifacts/model_pipeline.joblib

data/artifacts/model_metadata.joblib

In [ ]:
artifacts = train_model(
    processed_path="../data/processed/base_processada_reduzida_ML.parquet",
    save_artifacts=True,
)

print("Artefatos gerados com sucesso.")
print(artifacts["metrics"].keys())

## Conclusão

O modelo preditivo foi treinado com base na base processada reduzida de Machine Learning, contendo atributos acadêmicos e indicadores de desempenho dos alunos.

As etapas de feature engineering, separação treino/teste, modelagem e avaliação foram executadas com sucesso, gerando um artefato reutilizável para inferência em aplicação Streamlit.

Esse modelo pode ser incorporado a uma solução analítica para apoiar a identificação de alunos com risco de defasagem, contribuindo para ações preventivas e acompanhamento educacional.